# 004 · Encapsulation and Naming Conventions

> **Syntax › Classes & OOP** · Python 3.14

Python has no `public`, `private`, `protected` or `internal` keywords. Everything is public, and visibility is communicated through **naming conventions** that every Python developer — and every linter — understands.

**In this notebook you will learn:**

- the "we're all consenting adults" philosophy behind Python's lack of access modifiers
- the `_single_underscore` convention for internal members
- `__double_underscore` name mangling, and what it is really for
- why `__dunder__` names are special, not private
- `__all__` for controlling `from module import *`
- `typing.final` for "sealed", and options for read-only data

## C# mental model

| C# | Python |
|---|---|
| `public` | plain name: `balance` |
| `protected` / `internal` / `private` (by convention) | single underscore: `_balance` |
| `private` (to avoid clashes with subclasses) | double underscore: `__balance` → name-mangled |
| (no equivalent) | `__init__`, `__repr__` … "dunder" names reserved for the language |
| `sealed class` / `sealed override` | `@typing.final` (checked by type checkers only) |
| `readonly` field / `init` accessor | getter-only `@property`, or `@dataclass(frozen=True)` |

The key mental shift: Python's encapsulation is **advisory**, not enforced. The underscore tells other developers "this is an implementation detail, use at your own risk" — and tools (IDEs, linters, type checkers) back that up with warnings.

## "We're all consenting adults here"

This phrase from the Python community sums up the philosophy: the language trusts developers not to misuse internals, rather than physically preventing access. In practice this is less different from C# than it seems — C# `private` members can be reached via reflection too; Python just makes it easier and more honest.

Everything without an underscore is part of the public API:

```csharp
// C#
public class Account
{
    public string Owner { get; }
    private decimal _balance;
}
```

In [ ]:
class Account:
    def __init__(self, owner):
        self.owner = owner       # public
        self._balance = 0        # internal by convention

    def deposit(self, amount):
        self._validate(amount)
        self._balance += amount

    def _validate(self, amount): # internal helper method
        if amount <= 0:
            raise ValueError("amount must be positive")

acc = Account("alice")
acc.deposit(10)
print(acc.owner)

## `_single_underscore` — "internal, please don't touch"

A leading single underscore means *not part of the public API*. It covers the roles that `private`, `protected` and `internal` play in C#: subclasses and code in the same package may use it; external callers should not.

Python does not stop you — the attribute is fully accessible:

In [ ]:
print(acc._balance)       # works — but your IDE/linter will flag it
acc._balance = 1_000_000  # also works; you have been warned
print(acc._balance)

The convention has one runtime effect: names starting with `_` are skipped by `from module import *`. Linters such as Ruff/Pylint report accesses to `_private` members from outside the class (rule "protected-access").

## `__double_underscore` — name mangling

A name with **two leading underscores** (and at most one trailing underscore) inside a class body is *name-mangled*: Python rewrites `__secret` to `_ClassName__secret`. Accessing `obj.__secret` from outside therefore fails:

In [ ]:
class Vault:
    def __init__(self):
        self.__secret = "hunter2"   # stored as _Vault__secret

    def reveal(self):
        return self.__secret        # inside the class, the short name works

v = Vault()
print(v.reveal())

try:
    v.__secret
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

print(v.__dict__)                   # {'_Vault__secret': 'hunter2'}
print(v._Vault__secret)             # the mangled name is still accessible

⚠️ **Gotcha:** name mangling is **not** a privacy mechanism. Its real purpose is to prevent **accidental name clashes in subclasses**. Each class gets its own mangled name, so a parent's `__x` and a child's `__x` are two different attributes:

In [ ]:
class Base:
    def __init__(self):
        self.__id = "base-id"       # _Base__id

    def base_id(self):
        return self.__id

class Child(Base):
    def __init__(self):
        super().__init__()
        self.__id = "child-id"      # _Child__id — does NOT overwrite Base's

c = Child()
print(c.base_id())                  # base-id — Base's private state is intact
print(c.__dict__)

With a single underscore (`self._id`), the child would silently overwrite the parent's value. Use `__double` only when you are designing a class meant to be subclassed by others and you need that protection. For everything else, a single underscore is the idiomatic choice.

## Dunder names are special, not private

Names with two underscores on **both** sides — `__init__`, `__repr__`, `__len__`, `__eq__` — are called *dunder* ("double underscore") or *magic* methods. They are **public** hooks that the language itself calls, and they are **not** mangled. Never invent your own dunder names; that namespace is reserved for Python.

In [ ]:
class Box:
    def __init__(self, items):
        self.items = items

    def __len__(self):             # public protocol method — not private!
        return len(self.items)

b = Box([1, 2, 3])
print(len(b))                      # Python calls b.__len__()
print(b.__len__())                 # works, but call len(b) in real code

Dunder methods are covered in depth in *Classes & OOP › 007 · Dunder Methods*.

Here is a quick reference to all the underscore patterns:

| Name | Meaning |
|---|---|
| `name` | public |
| `_name` | internal (convention); skipped by `import *` |
| `__name` | name-mangled to `_Class__name`; avoids subclass clashes |
| `__name__` | special/dunder — defined by Python, public |
| `name_` | avoids clashing with a keyword: `class_`, `type_`, `from_` |
| `_` | throwaway variable: `for _ in range(3)` |

## Module-level privacy and `__all__`

The same conventions apply to modules: functions and variables starting with `_` are internal. A module can also define `__all__`, a list of the names exported by `from module import *`. It is the closest thing to choosing what is `public` vs `internal` at the assembly level, and it also documents the module's public API. *(see Modules & Packages › 002 · Creating Modules and Packages)*

In [ ]:
import sys
import tempfile
from pathlib import Path

source = '''
__all__ = ["greet"]

def greet(name):
    return f"Hello, {_decorate(name)}"

def helper():          # public name, but not listed in __all__
    return "helper"

def _decorate(name):   # internal
    return name.title()
'''

with tempfile.TemporaryDirectory() as tmp:
    Path(tmp, "greetings.py").write_text(source)
    sys.path.insert(0, tmp)
    try:
        ns = {}
        exec("from greetings import *", ns)
        print([n for n in ns if not n.startswith("__")])   # ['greet'] only

        import greetings
        print(greetings.helper())    # explicit access still works
    finally:
        sys.path.remove(tmp)
        sys.modules.pop("greetings", None)

## Sealed classes and methods — `typing.final`

C#'s `sealed` prevents inheritance (or further overriding). Python's equivalent is the `@final` decorator from `typing`. It is **only enforced by static type checkers** (mypy, pyright) — at runtime, subclassing still works.

```csharp
// C#
public sealed class Config { }
public class Base { public virtual void Run() { } }
public class Derived : Base { public sealed override void Run() { } }
```

In [ ]:
from typing import final

@final
class Config:
    pass

class Base:
    @final
    def run(self):
        return "Base.run"

class Hacked(Config):      # a type checker reports: cannot inherit from final class
    pass

print(Hacked())            # ...but Python itself happily runs it
print(Config.__final__)    # @final sets this marker attribute (since 3.11)

For overriding rules and `typing.override`, see *Classes & OOP › 005 · Inheritance and super()*.

## Read-only state

Python has no `readonly` keyword. The common options are:

1. **Getter-only property** over an underscore attribute *(see Classes & OOP › 003 · Properties)*.
2. **Frozen dataclass** — `@dataclass(frozen=True)` blocks assignment after construction, like a C# `record` with `init` accessors *(see Classes & OOP › 008 · Dataclasses)*.
3. **`NamedTuple`** or a plain `tuple` for small immutable records.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Money:
    amount: int
    currency: str

price = Money(10, "EUR")
try:
    price.amount = 20
except AttributeError as e:          # FrozenInstanceError subclasses AttributeError
    print(f"{type(e).__name__}: {e}")

Even these can be bypassed by determined code (e.g. `object.__setattr__`), consistent with the consenting-adults philosophy — they protect against **mistakes**, not malice.

## Summary

| Concept | C# | Python |
|---|---|---|
| Public member | `public` | `name` |
| Internal / protected / private | `internal` / `protected` / `private` | `_name` (convention) |
| Subclass-safe private | `private` | `__name` (mangled to `_Class__name`) |
| Language hooks | `override ToString()`, operators | `__dunder__` methods (public) |
| Module/assembly exports | `public` vs `internal` types | `__all__` + `_` prefix |
| Sealed class / method | `sealed` | `@typing.final` (type checkers only) |
| Read-only data | `readonly`, `init`, records | getter-only `@property`, `frozen=True` dataclass |
| Enforcement | compiler | conventions + linters/type checkers |

## Exercises

1. Write a `Thermostat` class whose current temperature is stored internally as `_celsius`, exposed only through a read-only `celsius` property, and changed via a `set_target(value)` method that validates the range 5–30.
2. Create a class with `self.__token = "abc"`. Print the instance's `__dict__` and access the token from outside the class using its mangled name.
3. Explain why `__init__` is **not** name-mangled while `__init` would be.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
class Thermostat:
    def __init__(self, celsius=20):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    def set_target(self, value):
        if not 5 <= value <= 30:
            raise ValueError("target must be between 5 and 30")
        self._celsius = value

# 2
class Session:
    def __init__(self):
        self.__token = "abc"

s = Session()
print(s.__dict__)           # {'_Session__token': 'abc'}
print(s._Session__token)    # abc

# 3 — mangling only applies to names with two leading underscores and
#     NOT ending in two underscores. __init__ ends with "__", so it is a
#     dunder (public, special) name; __init would become _Class__init.
```

</details>